In [1]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score

In [2]:
nltk.download('stopwords')
stopWords = set(stopwords.words('english'))
ps = PorterStemmer()
pattern = re.compile(r'<.*?>')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yajat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
data = pd.read_csv('IMDB Dataset.csv')
data = data.iloc[:10000]

In [4]:
def preprocess_text(text):
    text = re.sub(pattern, '', text)                         
    text = text.lower()                                     
    text = re.sub(r"[^\w\s']", ' ', text)                    
    words = [ps.stem(word) for word in text.split() if word not in stopWords] 
    return ' '.join(words)

In [5]:
data['review'] = data['review'].apply(preprocess_text)

In [6]:
cv = CountVectorizer(max_features=5000)
X = cv.fit_transform(data['review']).toarray()

In [7]:
data.rename(columns={'sentiment': 'label'}, inplace=True)
data['label'] = data['label'].map({'positive': 1, 'negative': 0})
y = data['label'].values

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
clf1 = GaussianNB()
clf2 = MultinomialNB()
clf3 = BernoulliNB()

clf1.fit(X_train, y_train)
clf2.fit(X_train, y_train)
clf3.fit(X_train, y_train)

BernoulliNB()

In [10]:
pred1 = clf1.predict(X_test)
pred2 = clf2.predict(X_test)
pred3 = clf3.predict(X_test)

In [11]:
print("Gaussian Accuracy:", accuracy_score(y_test, pred1))
print("Multinomial Accuracy:", accuracy_score(y_test, pred2))
print("Bernoulli Accuracy:", accuracy_score(y_test, pred3))

Gaussian Accuracy: 0.7195
Multinomial Accuracy: 0.845
Bernoulli Accuracy: 0.8535


In [16]:
def predict_sentiment(new_review_text):
    cleaned_input = preprocess_text(new_review_text)
    
    input_vector = cv.transform([cleaned_input]).toarray()
    
    prediction = clf3.predict(input_vector)
    
    return "Positive" if prediction[0] == 1 else "Negative"

In [17]:
sample_review_1 = "This movie was absolutely incredible! The acting and storyline were top notch."
sample_review_2 = "Terrible plot, awful pacing, and completely uninteresting characters."

print("Testing Custom Inputs\n")
print(f"Review 1: '{sample_review_1}'\nSentiment: {predict_sentiment(sample_review_1)}\n")
print(f"Review 2: '{sample_review_2}'\nSentiment: {predict_sentiment(sample_review_2)}")

Testing Custom Inputs

Review 1: 'This movie was absolutely incredible! The acting and storyline were top notch.'
Sentiment: Positive

Review 2: 'Terrible plot, awful pacing, and completely uninteresting characters.'
Sentiment: Negative
